# 06 — Synthèse et énoncé de la problématique

**Objectif** — rassembler les résultats des notebooks 01 à 05 en un énoncé de
problème défendable, et délimiter ce que ces données permettent ou non de conclure.

| | |
|---|---|
| **Entrée** | tables produites par les notebooks 01 à 05 |
| **Sortie** | `resultats/tables/06_chiffres_cles.csv` |

Ce notebook ne produit aucun résultat nouveau : il recalcule les chiffres clés à
partir de la source pour qu'ils soient **cohérents entre eux** et vérifiables en
un seul endroit.

In [1]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE / "src"))

import pandas as pd

from reclamations import chargement, config, texte

pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 70)

df = chargement.typer(chargement.charger_brut())
op = chargement.perimetre_operationnel(df)
analyse = chargement.perimetre_analyse(df)
analyse["texte"] = texte.construire_texte(analyse)
base = analyse[texte.a_texte_exploitable(analyse["texte"])].copy()
base["famille"] = texte.classer(base["texte"])

C:\Users\agent_dri_02\Desktop\data_alan\reclamation\niv\src\reclamations\texte.py:180: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  correspond = texte.str.contains(motif, regex=True, na=False)


## 1. Chiffres clés

In [2]:
suivi = op.sort_values(["contact_id", "date_creation"]).copy()
suivi["delai_j"] = (
    suivi.groupby("contact_id")["date_creation"].diff().dt.total_seconds() / 86400
)
suivi["redepot"] = (suivi["delai_j"] < config.FENETRE_REDEPOT_JOURS) & (
    suivi["ticket_type_name"] == suivi.groupby("contact_id")["ticket_type_name"].shift()
)
par_client = op["contact_id"].value_counts()
pct_non_classe = (base["famille"] == texte.NON_CLASSE).mean()
pct_regle = (base["famille"] == "debit_non_credit").mean() * 100
pct_debit_inj = (base["famille"] == "debit_injustifie").mean() * 100

cles = pd.DataFrame(
    [
        ("Périmètre", "Tickets, période opérationnelle", f"{len(op):,}", "nb 02"),
        ("Périmètre", "Tickets, périmètre d'analyse causale", f"{len(analyse):,}", "nb 03"),
        ("Périmètre", "Base textuelle exploitable", f"{len(base):,}", "nb 03"),
        ("Périmètre", "Couverture texte du périmètre d'analyse", f"{len(base) / len(analyse) * 100:.1f} %", "nb 03"),
        ("Qualité", "Tickets exclus — collecte dégradée (mars-mai 26)", f"{len(op) - len(analyse):,}", "nb 02"),
        ("Qualité", "Messages perdus par troncature à l'export", f"{op['nb_messages'].sum():,.0f}", "nb 01"),
        ("Cause", "Famille dominante (règles seules)", f"{pct_regle:.1f} %", "nb 04"),
        ("Cause", "Famille dominante (corrigée par audit 8/20)", f"~{pct_regle + pct_non_classe * 100 * 0.40:.0f} %", "nb 04"),
        ("Cause", "Argent prélevé à tort ou jamais arrivé (cumul corrigé)", f"~{pct_regle + pct_debit_inj + pct_non_classe * 100 * 0.80:.0f} %", "nb 04"),
        ("Cause", "Textes non reconnus par les règles", f"{pct_non_classe * 100:.1f} %", "nb 04"),
        ("Traitement", "Statut « resolved »", f"{(op['ticket_state_category'] == 'resolved').mean() * 100:.1f} %", "nb 05"),
        ("Traitement", "Messages par ticket (médiane)", f"{op['nb_messages'].median():.0f}", "nb 05"),
        ("Traitement", "Messages échangés / agent / mois", f"{op['nb_messages'].sum() / op['mois'].nunique() / op['admin_assignee_id'].nunique():,.0f}", "nb 05"),
        ("Récidive", "Clients ayant déposé >= 2 réclamations", f"{(par_client >= 2).mean() * 100:.1f} %", "nb 05"),
        ("Récidive", "Part du volume portée par ces clients", f"{par_client[par_client >= 2].sum() / par_client.sum() * 100:.1f} %", "nb 05"),
        ("Récidive", "Re-dépôts (< 7 j, même motif)", f"{suivi['redepot'].mean() * 100:.1f} %", "nb 05"),
    ],
    columns=["axe", "indicateur", "valeur", "source"],
)
chargement.sauver_table(cles, "06_chiffres_cles", index=False)
cles

  -> resultats\tables\06_chiffres_cles.csv


,axe,indicateur,valeur,source
0,Périmètre,"Tickets, période opérationnelle","18,056",nb 02
1,Périmètre,"Tickets, périmètre d'analyse causale","6,867",nb 03
2,Périmètre,Base textuelle exploitable,"6,545",nb 03
3,Périmètre,Couverture texte du périmètre d'analyse,95.3 %,nb 03
4,Qualité,Tickets exclus — collecte dégradée (mars-mai 26),"11,189",nb 02
5,Qualité,Messages perdus par troncature à l'export,"501,399",nb 01
6,Cause,Famille dominante (règles seules),44.5 %,nb 04
7,Cause,Famille dominante (corrigée par audit 8/20),~61 %,nb 04
8,Cause,Argent prélevé à tort ou jamais arrivé (cumul corrigé),~80 %,nb 04
9,Cause,Textes non reconnus par les règles,40.9 %,nb 04


## 2. Ce que les données établissent

**a. Le pic de mars-avril 2026 n'est pas un incident produit isolé.**
Quatre champs de saisie indépendants s'effondrent ensemble le 13 mars et
reviennent ensemble mi-mai. La signature est celle d'une mise en production, pas
d'un afflux de réclamations. *(notebook 02)*

**b. Le motif est massivement concentré.**
Environ six réclamations sur dix décrivent le même incident — un compte débité,
un bénéficiaire jamais crédité, sur un transfert SARA → mobile money. En y
ajoutant les débits injustifiés ou doublés, environ huit sur dix portent sur de
l'argent prélevé à tort ou jamais arrivé. *(notebook 04)*

**c. Une réclamation sur seize est une erreur de saisie du client**, corrigeable
par l'interface seule — afficher le nom du titulaire avant validation.
*(notebook 04)*

**d. Le dispositif de traitement produit une partie de sa propre charge.**
20 % du volume sont des re-dépôts, et ce taux monte à 30 % quand le système est
sous tension. *(notebook 05)*

## 3. Ce que les données ne peuvent pas établir

C'est la section la plus importante du rapport : elle délimite ce qui peut être
affirmé.

**Intercom est un journal de symptômes. Il contient le numérateur, jamais le
dénominateur.**

On sait que ~2 000 clients se sont plaints d'un transfert échoué. On ignore si
c'est sur 20 000 transferts (10 % d'échec — situation critique) ou sur 2 000 000
(0,1 % — bruit de fond normal). **Le même fichier est compatible avec les deux
situations, qui appellent des décisions opposées.**

Sont également hors de portée de cet export :

| Question | Pourquoi elle est hors de portée |
|---|---|
| Où la transaction casse-t-elle ? | Aucune trace technique : ni code d'erreur, ni étape, ni statut de la passerelle |
| Orange Money est-il plus défaillant que MTN ? | Les mentions sont confondues avec les parts de marché ; pas de volume par opérateur |
| L'argent est-il perdu ou en suspens ? | Aucune issue de dossier n'est enregistrée |
| Combien de clients subissent l'incident sans se plaindre ? | Par construction absents du fichier — et ce sont eux qui partent sans rien dire |
| Le délai de traitement s'améliore-t-il ? | `updated_at` est un proxy, pas une date de clôture ; 90 % des tickets n'ont pas de statut final |

### Données à demander

1. **Journal des transactions SARA** — identifiant, montant, opérateur, sens,
   horodatage, statut technique, code d'erreur. Joignable aux réclamations par
   montant + numéro + date. *C'est cette jointure qui transforme l'étude en
   diagnostic.*
2. **Ré-extraction d'Intercom avec `ticket_parts` complet** — un demi-million
   d'échanges tronqués à 4 caractères dans l'export actuel.
3. **Date de clôture et issue du dossier** (remboursé / rejeté / sans suite).

## 4. Énoncé de la problématique

> **Pourquoi une part significative des transferts entre SARA et les opérateurs
> de mobile money débite le client sans créditer le bénéficiaire — et comment
> agir à la source pour réduire le taux d'incident par transaction, plutôt que
> le nombre de plaintes ?**

### Le piège de mesure à écarter explicitement

Si l'indicateur de succès devient *le nombre de réclamations*, alors le moyen le
plus efficace de réussir est de **rendre la réclamation plus difficile**. Ce
levier n'est pas théorique : le notebook 02 montre que le canal a changé de
comportement en mars 2026 et que le volume en a été bouleversé, sans qu'aucune
réclamation réelle n'ait été résolue pour autant.

Le mécanisme joue dans les deux sens — un bon canal de réclamation **augmente**
les réclamations enregistrées, en révélant une insatisfaction jusque-là
silencieuse.

> **Indicateur retenu : le taux d'incident par transaction.** Il exige le
> dénominateur (journal des transactions), ce qui est précisément la donnée
> à demander.

### Décomposition en questions traitables

| # | Question | Faisable avec les données actuelles ? |
|---|---|---|
| 1 | Quelle est la structure réelle des motifs ? | **Oui** — classification apprise sur 6 545 textes, pour passer de 59 % à ~95 % de couverture |
| 2 | Quelle part est un re-dépôt évitable ? | **Oui** — déjà chiffrée à 20 %, à industrialiser en détection à l'ouverture |
| 3 | À quelle étape la transaction casse-t-elle ? | **Non** — nécessite le journal des transactions |
| 4 | Quels dossiers vont s'enliser ? | **Non** — pas de date de clôture ni d'issue de dossier |

### Prochaine étape

La question 1 est le premier livrable : remplacer les règles du notebook 04 par
une classification apprise, sur la base délimitée au notebook 03. C'est le seul
point de la problématique qui soit entièrement traitable avec les données
disponibles — et il conditionne la mesure de tous les autres.